# Practice: the grounding reward

The reward turns one episode into a number in [0, 1]: parse the final reply, keep only cites the model actually retrieved, check each claim against its cited evidence, measure coverage of the gold points, subtract a penalty for unsupported claims, and handle abstention by the item's `answerable` label. `docs/reward_spec.md` defines it.

This notebook builds small traces with the shared test builders, scores them with the real reward, and then scores them again with a **copy** of the aggregate defined here, so you can change λ, the partial-flag credit κ, or the formula itself and watch scores move. It never edits `src/`: it calls the public `neuro_grounded_qa` API and imports the builders from `tests/`.

Everything is synthetic: the fixture store has invented IDs like `GENE_A`, `DIS_1`, `E:000001`.

In [ ]:
import json
import sys
from pathlib import Path

# The builders live in tests/; put tests/ on the path, as pytest does.
REPO = Path.cwd() if (Path.cwd() / "tests").exists() else Path.cwd().parent
sys.path.insert(0, str(REPO / "tests"))

from contracts.builders import TraceBuilder, call, edge_payload, fixture_store
from neuro_grounded_qa.contracts import Abstain, Answer, QuestionItem, parse_output, retrieved_evidence
from neuro_grounded_qa.reward import (
    RewardParams, check_claims, coverage, evaluate, flagged, score, unsupported_penalty,
)

STORE = fixture_store()
ITEMS = {
    item.id: item
    for item in (
        QuestionItem.model_validate_json(line)
        for line in (REPO / "tests/fixtures/contracts/items.jsonl").read_text().splitlines()
        if line.strip()
    )
}
SINGLE, PARTIAL, UNANS = ITEMS["fx-single-001"], ITEMS["fx-partial-001"], ITEMS["fx-unans-001"]
print(SINGLE.prompt_text)
for gp in SINGLE.gold_points:
    print(" gold:", gp.subject_id, gp.relation, gp.object_id, gp.evidence)

## 1. Building episodes

`episode(item, final, retrieved)` makes a one-branch trace: the prompt, one sampled `get_edge` call per retrieved edge (each answered by a real tool result), then the model's final reply. `claim(...)` and `answer(...)` write the final reply as the model would.

In [ ]:
def claim(s, r, o, *cites):
    return {"subject_id": s, "relation": r, "object_id": o, "cites": list(cites)}

def answer(*claims, unsupported=()):
    body = {"claims": list(claims), "answer_text": "see claims"}
    if unsupported:
        body["unsupported_parts"] = list(unsupported)
    return json.dumps(body)

ABSTAIN = json.dumps({"abstain": True, "reason": "the evidence doesn't support an answer"})

def episode(item, final, retrieved=()):
    b = TraceBuilder(data=item)
    b.system("Answer only from retrieved evidence.")
    b.user(item.prompt_text)
    for i, eid in enumerate(retrieved):
        b.assistant(None, [call(f"call_{i}", "get_edge", edge_id=eid)])
        b.tool(f"call_{i}", edge_payload(eid), name="get_edge")
    b.assistant(final)
    return b.build()

GP1 = claim("GENE_A", "associated_with", "DIS_1", "E:000001")
GP2 = claim("GENE_B", "associated_with", "DIS_1", "E:000002")

## 2. Four answers to the single-hop question

- **honest**: both gold points, plus one honest mistake (cites a real edge that says something else).
- **padded**: the honest answer plus four true, retrieved, irrelevant claims.
- **fabricated**: cites `E:000002` without ever retrieving it.
- **abstain**: declines.

Plus the **evidence dump**: every edge touching `DIS_1` restated as a claim.

In [ ]:
MISTAKE = claim("GENE_C", "associated_with", "DIS_1", "E:000009")  # E:000009 is GENE_C - DIS_2
PADDING = [
    claim("GENE_A", "member_of", "PATH_P", "E:000007"),
    claim("GENE_C", "member_of", "PATH_P", "E:000008"),
    claim("GENE_C", "associated_with", "DIS_2", "E:000009"),
    claim("GENE_A", "interacts_with", "GENE_C", "E:000011"),
]
DIS_1_EDGES = STORE.neighbors("DIS_1", relation=None, limit=50)

EPISODES = {
    "honest": episode(SINGLE, answer(GP1, GP2, MISTAKE), ["E:000001", "E:000002", "E:000009"]),
    "padded": episode(
        SINGLE, answer(GP1, GP2, MISTAKE, *PADDING),
        ["E:000001", "E:000002", "E:000007", "E:000008", "E:000009", "E:000011"],
    ),
    "fabricated": episode(SINGLE, answer(GP1, GP2), ["E:000001"]),
    "abstain": episode(SINGLE, ABSTAIN, ["E:000001"]),
    "evidence dump": episode(
        SINGLE,
        answer(*[claim(e.subject.id, e.relation, e.object.id, e.id) for e in DIS_1_EDGES]),
        [e.id for e in DIS_1_EDGES],
    ),
}

def row(name, b):
    cov = "n/a" if b.coverage is None else f"{b.coverage:.2f}"
    return (f"{name:14s} reward={b.reward:.3f}  cov={cov}  S={b.n_supported} U={b.n_unsupported}"
            f"  penalty={b.penalty:.3f}  outcome={b.outcome}")

for name, trace in EPISODES.items():
    print(row(name, evaluate(trace, SINGLE).breakdown))

Things to notice:

- The fabricated cite is stripped: the claim it backed is unsupported, so the answer gets half coverage and pays a penalty.
- **Padding beats honest.** The padding is true and supported, so it can't add coverage, but it dilutes the penalty share: λ·1/7 instead of λ·1/3. That's the penalty-dilution exploit candidate in the spec.
- **The evidence dump scores 1.0.** Every claim is supported, so there's no penalty, and the gold points are among the neighbors. The brief lists this as the open problem for the audit.

## 3. A copy of the aggregate to experiment on

`my_aggregate` below is a copy of `neuro_grounded_qa.reward.scoring.aggregate`, and `my_score` wires it to the real components. Edit either one here; `src/` is untouched.

In [ ]:
def my_aggregate(answerable, outcome, cov, penalty, is_flagged, n_claims,
                 lam=0.5, kappa=0.2, correctable_abstain=1.0, flag_only_counts=True):
    if outcome == "parse_failure":
        return 0.0
    abstained = outcome == "abstain"
    if answerable == "yes":
        return 0.0 if abstained else (cov or 0.0) - penalty
    if answerable == "partial":
        if abstained:
            return 0.0
        flag = 1.0 if is_flagged and (n_claims > 0 or flag_only_counts) else 0.0
        return (1 - kappa) * (cov or 0.0) + kappa * flag - penalty
    if answerable == "no":
        return 1.0 if abstained else 0.0
    if answerable == "correctable":
        return correctable_abstain if abstained else (cov or 0.0) - penalty
    raise ValueError(answerable)


def my_score(item, trace, aggregate=my_aggregate, **params):
    parsed = parse_output(trace.last_reply)
    retrieved = retrieved_evidence(trace)
    if isinstance(parsed, Answer):
        checks = check_claims(parsed, retrieved, item)
        cov, _ = coverage(item, checks)
        is_flagged = flagged(parsed)
        outcome = "answer"
    else:
        checks, cov, is_flagged = (), (0.0 if item.gold_points else None), False
        outcome = "abstain" if isinstance(parsed, Abstain) else "parse_failure"
    n_sup = sum(c.supported for c in checks)
    penalty = unsupported_penalty(n_sup, len(checks) - n_sup, params.get("lam", 0.5))
    raw = aggregate(item.answerable, outcome, cov, penalty, is_flagged, len(checks), **params)
    return min(1.0, max(0.0, raw))


# The copy agrees with the real reward at the default constants.
for name, trace in EPISODES.items():
    assert abs(my_score(SINGLE, trace) - evaluate(trace, SINGLE).breakdown.reward) < 1e-12, name
print("copy matches the package")

## 4. Sweep λ

Watch the honest answer fall as λ rises, while the padded answer falls more slowly (dilution) and the evidence dump doesn't move at all.

In [ ]:
lams = [0.0, 0.25, 0.5, 1.0, 2.0]
print(f"{'':14s}" + "".join(f"λ={lam:<6}" for lam in lams))
for name, trace in EPISODES.items():
    print(f"{name:14s}" + "".join(f"{my_score(SINGLE, trace, lam=lam):<8.3f}" for lam in lams))

## 5. Sweep κ on the partial item

The partial item asks two things; only the first is in the graph. Compare a complete answer with and without the flag, a half answer with the flag, a flag-only answer and an abstention.

In [ ]:
FLAG = ["no drug acting on Gene Delta is in the graph"]
PARTIAL_EPISODES = {
    "full + flag": episode(PARTIAL, answer(GP1, GP2, unsupported=FLAG), ["E:000001", "E:000002"]),
    "full, no flag": episode(PARTIAL, answer(GP1, GP2), ["E:000001", "E:000002"]),
    "half + flag": episode(PARTIAL, answer(GP1, unsupported=FLAG), ["E:000001"]),
    "flag only": episode(PARTIAL, answer(unsupported=FLAG)),
    "abstain": episode(PARTIAL, ABSTAIN),
}
kappas = [0.0, 0.1, 0.2, 0.4]
print(f"{'':14s}" + "".join(f"κ={k:<6}" for k in kappas))
for name, trace in PARTIAL_EPISODES.items():
    print(f"{name:14s}" + "".join(f"{my_score(PARTIAL, trace, kappa=k):<8.3f}" for k in kappas))

## 6. Swap the formula: the brief's additive partial credit

The brief's wording reads as `coverage + κ·flag − penalty`, clipped. Under that form, a complete answer scores 1 with or without the flag. Compare it with the weighted form the package uses.

In [ ]:
def additive_aggregate(answerable, outcome, cov, penalty, is_flagged, n_claims, kappa=0.2, **kw):
    if answerable == "partial" and outcome == "answer":
        return (cov or 0.0) + kappa * float(is_flagged) - penalty
    return my_aggregate(answerable, outcome, cov, penalty, is_flagged, n_claims, kappa=kappa, **kw)

print(f"{'':14s}{'weighted':>10s}{'additive':>10s}")
for name, trace in PARTIAL_EPISODES.items():
    w = my_score(PARTIAL, trace)
    a = my_score(PARTIAL, trace, aggregate=additive_aggregate)
    print(f"{name:14s}{w:>10.3f}{a:>10.3f}")

## 7. Always-abstain over the target type mix

The brief's target mix is single-hop 30%, multi-hop 25%, partial 15%, unanswerable 20%, false premise 10% (split evenly here between the `no` and `correctable` fixture items). Change the correctable-abstain credit and see how the floor moves.

In [ ]:
MIX = {"fx-single-001": 0.30, "fx-multi-001": 0.25, "fx-partial-001": 0.15,
       "fx-unans-001": 0.20, "fx-falsep-001": 0.05, "fx-falsep-002": 0.05}

for credit in (1.0, 0.7, 0.5):
    total = sum(w * my_score(ITEMS[i], episode(ITEMS[i], ABSTAIN), correctable_abstain=credit)
                for i, w in MIX.items())
    print(f"correctable abstain credit {credit}: always-abstain scores {total:.3f}")

## Things to try

- Give the evidence dump a cost: a cap on claims, or a precision term (supported gold-matching claims / all claims). Does the honest answer still beat it?
- Count duplicate claims once in the penalty and see what happens to padding with repeats.
- Make flags cost something on `yes` items.
- Require `same_fact` to respect direction for some relations (copy it here) and re-run section 2 with a reversed claim.